## Try adding morphology information to a redshift estimator 

In the notebook we are going to add size information to the features we use to train the estimator

#### Standard imports

In [ ]:
import os
import tables_io
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.ensemble import (HistGradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import HuberRegressor, LinearRegression, QuantileRegressor
from sklearn.svm import NuSVR
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn import preprocessing
from sklearn.decomposition import PCA
import plotting_functions

#### Change this to match your data location

In [ ]:
pz_dir = '.'

#### Read the test and training data

In [ ]:
train = tables_io.read(f'{pz_dir}/data/lsst_cat_matched_nonan_train.hdf5')
test = tables_io.read(f'{pz_dir}/data/lsst_cat_matched_nonan_test.hdf5')


#### Make a quality cut, and apply it to the data

In [ ]:
def make_mask(t):
    mask = (t['refExtendedness'] == 1) *\
           (t['i_blendedness'] < 0.01) *\
           (t['g_psfFlux']/ t['g_psfFluxErr'] > 10) *\
           (t['i_psfFlux'] / t['i_psfFluxErr'] > 10) *\
           (t['r_psfFlux'] / t['r_psfFluxErr'] > 10) *\
           (t['z_psfFlux'] / t['z_psfFluxErr'] > 10)
    return mask

In [ ]:
mask_test = make_mask(test)
mask_train = make_mask(train)

#### Extract the features we will use to train the algorithm

In [ ]:
def get_features(t):
    """ This will include the sersic fit index,    
    the trace in g and z bands, and the trace of the sersic fits
    in addition to the magntidues
    """
    sersic_x = t['sersic_reff_x']  
    sersic_y = t['sersic_reff_y']
    sersic_trace = np.nan_to_num(sersic_x*sersic_x + sersic_y*sersic_y, nan=0)
    sersic_index = np.nan_to_num(t['sersic_index'], 0)

    g_trace = np.nan_to_num(t['g_ixx'] + t['g_iyy'], nan=0)
    z_trace = np.nan_to_num(t['z_ixx'] + t['z_iyy'], nan=0)

    feature_list = [sersic_index, g_trace, z_trace, sersic_trace]
    feature_list += [np.nan_to_num(t[f'{band}_psfMag'], nan=30) for band in 'ugrizy']
    
    return np.vstack(feature_list).T


In [ ]:
features_test = get_features(test)[mask_test]
features_train = get_features(train)[mask_train]

targets_test = test['redshift'][mask_test]
targets_train = train['redshift'][mask_train]

#### Here is a utility function to run the redshift estimation

In [ ]:
def run_regression(
    regerssor,
    train_features: np.ndarray,
    train_targets: np.ndarray,
    test_features: np.ndarray,
) -> np.ndarray:
    # NOTE: breakpoint() left here for interactive debugging — remove before running in batch.
    breakpoint()
    # Standardize → clip outliers → PCA → fit → predict.
    # StandardScaler + PCA is especially important when mixing morphology features (moments
    # in pixel² units) with magnitudes (logarithmic), which have very different scales.
    scaler = preprocessing.StandardScaler().fit(train_features)
    scaled_train = scaler.transform(train_features).clip(-5, 5)    
    pca = PCA(n_components=train_features.shape[-1], whiten=False).fit(scaled_train)    
    pca_train = pca.transform(scaled_train)    
    scaled_test = scaler.transform(test_features).clip(-5, 5)
    pca_test = pca.transform(scaled_test)
    regerssor.fit(pca_train, train_targets)
    return regerssor.predict(pca_test)

#### Here are a bunch of different ML algorithms

In [ ]:
hbr = HistGradientBoostingRegressor()
etr = ExtraTreesRegressor()
abr = AdaBoostRegressor()
gpr = GaussianProcessRegressor()
isr = IsotonicRegression()
hur = HuberRegressor()
lir = LinearRegression()
qur = QuantileRegressor()
nsr = NuSVR()
knr = KNeighborsRegressor()
rnr = RadiusNeighborsRegressor()

#### Here are four utility functions that use different sub-sets of the training data

In [ ]:
def run_it(reg):
    """Use only the 6 PSF magnitudes (features[:,4:]) — no morphology."""
    preds = run_regression(reg, features_train[:,4:], targets_train, features_test[:,4:])
    _ = plotting_functions.plot_true_predict_fancy(targets_test, np.nan_to_num(preds, -0.4))

In [ ]:
def run_it_size(reg):
    """Add the Sersic trace size (features[:,3]) alongside the magnitudes."""    
    preds = run_regression(reg, features_train[:,3:], targets_train, features_test[:,3:])
    _ = plotting_functions.plot_true_predict_fancy(targets_test, np.nan_to_num(preds, -0.4))

In [ ]:
def run_it_sizes(reg):
    """Add g-band and z-band moment traces + Sersic trace (features[:,1:]) alongside magnitudes."""        
    preds = run_regression(reg, features_train[:,1:], targets_train, features_test[:,1:])
    _ = plotting_functions.plot_true_predict_fancy(targets_test, np.nan_to_num(preds, -0.4))

In [ ]:
def run_it_all(reg):
    """Full feature set: Sersic index + g/z traces + Sersic trace + 6 magnitudes (all of features[:,0:])."""            
    preds = run_regression(reg, features_train, targets_train, features_test)
    _ = plotting_functions.plot_true_predict_fancy(targets_test, np.nan_to_num(preds, -0.4))

In [ ]:
run_it(etr)

In [ ]:
run_it_size(etr)

In [ ]:
run_it_sizes(etr)

In [ ]:
run_it_all(etr)